# Finalizing the ABT 

**A usable ABT should meet a few requirements before modeling:**
1) **A clear unit of analysis**: One unit is easy to understand, makes business sense, and can be used to make an actionable decision. The movies_abt row unit is one movie with ratings aggregated.
2) **Unique key columns**: All key columns are unique with no duplicates. Duplicates would skew the results. Checks will validate that each movieId identifier contains no duplicate values.
3) **Reasonable values**: Value ranges make are within reason. Categorical features like genre and month are defined. Describe function used to evaluate columns.
4) **Target variable reviewed**: Clearly state the thresholds, how they are derived, and why they make business sense.
5) **Missing values understood**: All missing values documented and understood. 
6) **Columns cleary defined**: Every column name has meaning and data type, unit, and sources are understood.
7) **Transformation steps summarized**: There's an explanation of the sequence from raw inputs to the final ABT

Most of these steps were completed or mentioned in Milestone 2, so we will only briefly cover them in Milestone 3.


---
## Loading The Data

The initial step in this project is to load the required libraries and the datasets. 
 
The libraries required are...   
        - pathlib : a file handler that treats paths as objects rather than text strings.   
        - pandas : a library of functions that are used to manipulate dataframes.  
        - numpy : a library of math functions.  

Next, we define the root folder where the project folders and files will be saved.  

The project directory contains the following folders...  
        - The data folder : This folder contains raw data files.  
        - The output folder : This folder contains the final ABT and processed data files.    
        - The report folder : This folder contains html reports, graphs, and analysis.  



In [100]:
# This code loads libraries and sets directories for the project.
#  pathlib is a file handler. treats paths as objects not strings
# pandas brings in df functions
# numpy provides math functions.

#Checking my Python environment 
import sys; print(sys.executable); print(sys.version)

from pathlib import Path
import pandas as pd
import numpy as np

# Imports for Milestone 3 modeling
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score,
                             mean_absolute_error, mean_squared_error, r2_score)

#Set repository root to the current working directory. One step above readme location.

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

display(REPO_ROOT)

#Create directories for data, outputs, and reports if they don't exist.
DATA_DIR = REPO_ROOT / "data"
RAW_DATA = REPO_ROOT / "data/raw"
PROCESSED_DATA = REPO_ROOT / "outputs/processed"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)
RAW_DATA.mkdir(parents=True, exist_ok=True)

#set max column display to 100 columns for pandas dataframes.
pd.set_option("display.max_columns", 100)

#This code loads the raw data into dataframes.

# Load the datasets
movies = pd.read_csv(RAW_DATA / 'movies_metadata.csv')
ratings = pd.read_csv(RAW_DATA / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(RAW_DATA / 'links.csv')
credits = pd.read_csv(RAW_DATA / 'credits_curated.csv')
keywords = pd.read_csv(RAW_DATA / 'keywords_curated.csv')

# This code loads the processed data created in Milestone 2.

movie_cast = pd.read_csv(PROCESSED_DATA / 'movie_cast.csv')
movie_genres = pd.read_csv(PROCESSED_DATA / 'movie_genres.csv')
movie_keywords = pd.read_csv(PROCESSED_DATA / 'movie_keywords.csv')
movies_abt = pd.read_csv(PROCESSED_DATA / 'movies_abt.csv')
movies_clean = pd.read_csv(PROCESSED_DATA / 'movies_clean.csv')
ratings_aggregated = pd.read_csv(PROCESSED_DATA / 'ratings_aggregated.csv')
cleaning_log = pd.read_csv(PROCESSED_DATA / 'cleaning_log.csv')

print("Datasets Loaded")

c:\python314\python.exe
3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]


WindowsPath('c:/Users/matth/Project-Option-1-Matt-Flores')

Datasets Loaded


---
## Drafting the data dictionary

The data dictionary is a critical component when handing off the analysis. It is a document that shows each column in the abt along with data type, descriptions, source, trasnformation/derivation method, and notes or concerns.

In this section we will create the dictionary and add info for each column of the movies_abt

In [101]:
# Define the data dictionary as a dataframe with required attributes.
data_dictionary = pd.DataFrame({
    "column_name": movies_abt.columns,
    "data_type": [str(movies_abt[col].dtype) for col in movies_abt.columns],
    "description": "", "source": "",
    "transformation_or_derivation": "", "notes_or_concerns": "",
})

doc_fields = ["description", "source", "transformation_or_derivation", "notes_or_concerns"]

entries = {
    "movieId": ["Primary key for movies", "movies table", "none", "All unique"],
    "tmdbId": ["Key that joins credits and keywords", "movies table", "none", "All unique"],
    "imdb_id": ["IMDb identifier", "movies table", "none", "Not used"],
    "title": ["Movie title", "movies table", "none", "Duplicate titles exist; they are remakes or re-releases"],
    "original_language": ["Original language of the movie", "movies table", "Converted object to category", "Categorical data"],
    "release_date": ["Movie release date", "movies table", "Converted object to datetime64 (errors='coerce')", "No NaT after conversion; range 1902-09-01 to 2016-09-23"],
    "runtime": ["Movie runtime in minutes", "movies table", "none", "Might be used as a feature"],
    "budget": ["Movie budget (USD)", "movies table", "Converted int64 to float64; filtered to > 10,000", "Used for ROI calculation"],
    "revenue": ["Movie revenue (USD)", "movies table", "Filtered to > 10,000", "Used for ROI calculation. Leakage: post-release; builds the ROI target, do not use as a predictor"],
    "vote_average": ["TMDb average vote (10-point scale)", "movies table", "none", "Might be used as a target. Leakage: post-release; do not use as a predictor"],
    "vote_count": ["TMDb vote count", "movies table", "Converted float64 to Int64", "More votes per movie than the MovieLens rating_count. Leakage: post-release; do not use as a predictor"],
    "genres_list": ["Pipe-separated list of genres", "movies table", "none", "Categorical feature"],
    "primary_genre": ["Primary genre of the movie", "movies table", "Converted object to category", "Categorical feature"],
    "release_year": ["Year of release", "movies table", "none", "Exploratory, not a feature"],
    "roi": ["Movie's return on investment", "derived", "(revenue - budget) / budget", "ROI >= 1 is assumed success. Leakage: outcome, not known at planning; never a predictor"],
    "roi_threshold_met": ["ROI success flag", "derived", "roi >= 1 = 1, else 0", "Primary target. Leakage: outcome, not known at planning; never a predictor"],
    "rating_count": ["Number of MovieLens ratings", "ratings table (MovieLens)", "Count of ratings grouped by movieId", "Most movies have < 20 ratings. Leakage: post-release; do not use as a predictor"],
    "avg_rating": ["MovieLens mean rating (0.5-5 scale)", "ratings table (MovieLens)", "Mean rating grouped by movieId", "May be unreliable for movies with few ratings. Leakage: post-release; builds the rating target, do not use as a predictor"],
    "min_rating": ["MovieLens minimum rating", "ratings table (MovieLens)", "Min rating grouped by movieId", "May be unreliable for movies with few ratings. Leakage: post-release; do not use as a predictor"],
    "max_rating": ["MovieLens maximum rating", "ratings table (MovieLens)", "Max rating grouped by movieId", "May be unreliable for movies with few ratings. Leakage: post-release; do not use as a predictor"],
    "rating_std_dev": ["Standard deviation of MovieLens ratings", "ratings table (MovieLens)", "Std dev of rating grouped by movieId", "May be unreliable for movies with few ratings. Leakage: post-release; do not use as a predictor"],
    "distinct_users": ["Number of unique users who rated the movie", "ratings table (MovieLens)", "Distinct userId count grouped by movieId", "Used to validate one rating per user per movie. Leakage: post-release; do not use as a predictor"],
    "min_rating_date": ["Date of first MovieLens rating", "ratings table (MovieLens)", "Min rating_datetime grouped by movieId", "Kept for exploratory purposes. Leakage: post-release; do not use as a predictor"],
    "max_rating_date": ["Date of most recent MovieLens rating", "ratings table (MovieLens)", "Max rating_datetime grouped by movieId", "Kept for exploratory purposes. Leakage: post-release; do not use as a predictor"],
    "rating_threshold_met": ["MovieLens rating success flag", "derived", "avg_rating >= 4 = 1, else 0", "Secondary target. Leakage: outcome, not known at planning; never a predictor"],
    "director": ["Movie director", "credits table", "Left merge on tmdbId", "Kept for exploratory purposes"],
    "top_cast": ["Pipe-separated list of top cast", "credits table", "Left merge on tmdbId", "Good for exploration; might omit from features"],
    "keywords_list": ["Pipe-separated list of content keywords", "keywords table", "Left merge on tmdbId", "String to convert to dummy variables"],
    "release_month": ["Month of release (1-12)", "derived", "release_date.dt.month", "Categorical feature"],
}

for abt_column, values in entries.items():
    data_dictionary.loc[data_dictionary["column_name"] == abt_column, doc_fields] = values

dd_check = data_dictionary[doc_fields].fillna("")
for col in doc_fields:
    dd_check[col] = dd_check[col].str.strip()

print("These checks will return 0 unless there is a missing column in the data dictionary") 
print(len(set(movies_abt.columns) - set(data_dictionary["column_name"]))) 
print(dd_check.eq("").sum().sum())                                         

data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

display(data_dictionary)

These checks will return 0 unless there is a missing column in the data dictionary
0
0


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,Primary key for movies,movies table,none,All unique
1,tmdbId,int64,Key that joins credits and keywords,movies table,none,All unique
2,imdb_id,str,IMDb identifier,movies table,none,Not used
3,title,str,Movie title,movies table,none,Duplicate titles exist; they are remakes or re...
4,original_language,str,Original language of the movie,movies table,Converted object to category,Categorical data
5,release_date,str,Movie release date,movies table,Converted object to datetime64 (errors='coerce'),No NaT after conversion; range 1902-09-01 to 2...
6,runtime,float64,Movie runtime in minutes,movies table,none,Might be used as a feature
7,budget,float64,Movie budget (USD),movies table,"Converted int64 to float64; filtered to > 10,000",Used for ROI calculation
8,revenue,float64,Movie revenue (USD),movies table,"Filtered to > 10,000",Used for ROI calculation. Leakage: post-releas...
9,vote_average,float64,TMDb average vote (10-point scale),movies table,none,Might be used as a target. Leakage: post-relea...


---
## Review cleaning_log and movies_abt

Let's first refresh our memory of the cleaning log and movies abt created in milestone 2.

In [102]:
# display cleaning log for review
display(cleaning_log)

# display movies abt for review
display(movies_abt.head())

,method,table,column,description,reason,notes
0,conversion,movies,release_date,object to datetime64 (errors=coerce),enable date filtering and pandas date features,"no NaT. min date is 1902-09-01, max date is 20..."
1,conversion,movies,vote_count,float64 to Int64,show as whole number,no NaN. 44 movies have no votes
2,conversion,movies,budget,int64 to float64,match revenue dtype,no NaN. 48.9% of movies have 0 budget
3,conversion,movies,primary_genre,object to category,few unique values,20 unique values
4,conversion,movies,status,object to category,few unique values,4 unique values
5,conversion,movies,original_language,object to category,few unique values,42 unique values
6,aggregation,ratings_aggregated,"rating, userId, rating_datetime",grouped ratings metrics by movieId,transform to 1 row per movie,"3,042 movieIds have 1 review. A std dev can't ..."
7,derivation,ratings_aggregated,rating_threshold_met,created rating success flag,can be used for filtering or probability analysis,some flags may lack reliable counts
8,reshaping,movies,genres_list,split on | to create long data,one row per movie-genre pair,movie_genres table created kept separate from abt
9,reshaping,credits,top_cast,split on | to create long data,one row per movie-actor pair,movie_cast table created kept separate from abt


,movieId,tmdbId,imdb_id,title,original_language,release_date,runtime,budget,revenue,vote_average,vote_count,genres_list,primary_genre,release_year,roi,roi_threshold_met,rating_count,avg_rating,min_rating,max_rating,rating_std_dev,distinct_users,min_rating_date,max_rating_date,rating_threshold_met,director,top_cast,keywords_list,release_month
0,1,862,tt0114709,Toy Story,en,1995-10-30,81.0,30000000.0,373554033.0,7.7,5415,Animation|Comedy|Family,Animation,1995,11.451801,1,247.0,3.872470,1.0,5.0,0.958981,247.0,1996-03-30 19:00:13,2016-10-06 19:55:11,0.0,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,10
1,2,8844,tt0113497,Jumanji,en,1995-12-15,104.0,65000000.0,262797249.0,6.9,2413,Adventure|Fantasy|Family,Adventure,1995,3.043035,1,107.0,3.401869,1.5,5.0,0.880714,107.0,1996-03-30 19:12:30,2016-08-01 17:42:33,0.0,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,12
2,4,31357,tt0114885,Waiting to Exhale,en,1995-12-22,127.0,16000000.0,81452156.0,6.1,34,Comedy|Drama|Romance,Comedy,1995,4.090760,1,13.0,2.384615,1.0,3.5,0.938835,13.0,1996-06-10 16:45:35,2004-07-27 06:14:12,0.0,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,12
3,6,949,tt0113277,Heat,en,1995-12-15,170.0,60000000.0,187436818.0,7.7,1886,Action|Crime|Drama|Thriller,Action,1995,2.123947,1,104.0,3.884615,1.0,5.0,0.830928,104.0,1996-03-30 19:12:30,2016-08-16 22:34:44,0.0,Michael Mann,Al Pacino|Robert De Niro|Val Kilmer,robbery|detective|bank|obsession|chase|shootin...,12
4,9,9091,tt0114576,Sudden Death,en,1995-12-22,106.0,35000000.0,64350171.0,5.5,174,Action|Adventure|Thriller,Action,1995,0.838576,0,20.0,3.150000,1.0,5.0,0.812728,20.0,1996-08-15 09:28:40,2002-03-16 22:26:40,0.0,Peter Hyams,Jean-Claude Van Damme|Powers Boothe|Dorian Har...,terrorist|hostage|explosive|vice president,12


---
## Final Validation Checks

Final check to verify there are no duplicate keys, unreasonable values, and missing values are understood.

We will create a validation checklist dataframe with our findings.

In [103]:
# --- Duplicate key check ---
print("--- Duplicated Key Check ---")
display(movies_abt['movieId'].duplicated().sum())

# --- Unique values check ---
print("\n--- Unique values check ---")
display(movies_abt.nunique())

print("--- Checking if duplicated titles are valid ---")
display(movies_abt.loc[movies_abt.duplicated(subset='title', keep=False), ['movieId', 'title', 'release_date']]
    .sort_values(by=['title', 'release_date'], ascending=[True, False]).head(10))

# --- Reasonable values check ---
print("--- describe roi column ---")
display(movies_abt['roi'].describe())

print("--- describe budget column ---")
display(movies_abt['budget'].describe())

print("--- describe revenue column ---")
display(movies_abt['revenue'].describe())

print("--- describe runtime column ---")
display(movies_abt['runtime'].describe())

print("--- describe avg_rating column ---")
display(movies_abt['avg_rating'].describe())

print("--- describe vote_average column ---")
display(movies_abt['vote_average'].describe())

print(f"roi missing: {movies_abt['roi'].isna().sum()}, roi inf: {np.isinf(movies_abt['roi']).sum()}")

print("\n--- review top and bottom movies by roi column ---")
display(pd.concat([movies_abt.sort_values(by='roi', ascending=False).head(5),
    movies_abt.sort_values(by='roi', ascending=False).tail(5)]))

# --- Missing values check ---
print("--- isna Sums Check ---")
display(movies_abt.isna().sum())

print(f"movies with 1 rating: {movies_abt.query('rating_count == 1').shape[0]}")

dd_check = data_dictionary[doc_fields].fillna("")   # 1. copy the 4 text columns, NaN -> ""
for col in doc_fields:
    dd_check[col] = dd_check[col].str.strip()        # 2. remove spaces, so " " counts as blank

dd_check.eq("").sum().sum()                          # 3. count cells that are empty

validation_checklist = pd.DataFrame([
    {"check": "Duplicate keys", "result": movies_abt["movieId"].duplicated().sum()},
    {"check": "Missing key values", "result": movies_abt["movieId"].isna().sum()},
    {"check": "Missing ABT cells", "result": movies_abt.isna().sum().sum()},
    {"check": "Missing target values", "result": movies_abt["roi"].isna().sum()},
    {"check": "ABT columns without a dictionary row",
     "result": len(set(movies_abt.columns) - set(data_dictionary["column_name"]))},
    {"check": "Blank dictionary cells", "result": dd_check.eq("").sum().sum()},
])  # the lab also added "status" and "notes" keys to each row

display(validation_checklist)

--- Duplicated Key Check ---


np.int64(0)


--- Unique values check ---


movieId                 3813
tmdbId                  3813
imdb_id                 3813
title                   3750
original_language         26
release_date            2962
runtime                  144
budget                   459
revenue                 3670
vote_average              56
vote_count              1690
genres_list             1049
primary_genre             18
release_year              93
roi                     3775
roi_threshold_met          2
rating_count             185
avg_rating              1115
min_rating                10
max_rating                10
rating_std_dev          1950
distinct_users           185
min_rating_date         3465
max_rating_date         3649
rating_threshold_met       2
director                1572
top_cast                3738
keywords_list           3689
release_month             12
dtype: int64

--- Checking if duplicated titles are valid ---


,movieId,title,release_date
321,1019,"20,000 Leagues Under the Sea",1954-12-23
2782,62383,"20,000 Leagues Under the Sea",1916-12-24
3015,77798,A Nightmare on Elm Street,2010-04-30
455,1347,A Nightmare on Elm Street,1984-11-14
2988,74789,Alice in Wonderland,2010-03-03
329,1032,Alice in Wonderland,1951-07-03
3156,86293,Arthur,2011-04-08
1230,3524,Arthur,1981-07-17
3660,122888,Ben-Hur,2016-08-17
436,1287,Ben-Hur,1959-12-26


... (14 outputs left out)

--- isna Sums Check ---


movieId                   0
tmdbId                    0
imdb_id                   0
title                     0
original_language         0
release_date              0
runtime                   0
budget                    0
revenue                   0
vote_average              0
vote_count                0
genres_list               1
primary_genre             1
release_year              0
roi                       0
roi_threshold_met         0
rating_count             17
avg_rating               17
min_rating               17
max_rating               17
rating_std_dev          644
distinct_users           17
min_rating_date          17
max_rating_date          17
rating_threshold_met     17
director                  0
top_cast                  3
keywords_list            75
release_month             0
dtype: int64

movies with 1 rating: 627


,check,result
0,Duplicate keys,0
1,Missing key values,0
2,Missing ABT cells,860
3,Missing target values,0
4,ABT columns without a dictionary row,0
5,Blank dictionary cells,0


I decided to remove rows with missing top_cast or keywords_list values.

In [104]:
rows_before = len(movies_abt)

movies_abt = movies_abt.dropna(subset=["genres_list", "top_cast", "keywords_list"]).copy()

rows_after = len(movies_abt)
print(f"rows dropped for missing genre/cast/keywords: {rows_before - rows_after:,} ({rows_before:,} -> {rows_after:,})")

# --- Missing values check ---
print("--- isna Sums Check ---")
display(movies_abt.isna().sum())


rows dropped for missing genre/cast/keywords: 79 (3,813 -> 3,734)
--- isna Sums Check ---


movieId                   0
tmdbId                    0
imdb_id                   0
title                     0
original_language         0
release_date              0
runtime                   0
budget                    0
revenue                   0
vote_average              0
vote_count                0
genres_list               0
primary_genre             0
release_year              0
roi                       0
roi_threshold_met         0
rating_count             17
avg_rating               17
min_rating               17
max_rating               17
rating_std_dev          605
distinct_users           17
min_rating_date          17
max_rating_date          17
rating_threshold_met     17
director                  0
top_cast                  0
keywords_list             0
release_month             0
dtype: int64

---
## ROI Model: Target and feature explanation

Now that we've complete our final abt checks. We will begin constructing our roi model.

**Target (y)**: The roi model target is the roi_threshold_met classification.

**Features (X)**: We will use all columns possible, evaulate the coefficients and model performance, then make adjustments as needed. There are category and numeric features.

**Leakage**: We will omit any columns that would not be known during the movie planning phase. These include ratings related attributes and revenue.

In [105]:
# Set target
target = "roi_threshold_met"

# set feature columns
feature_columns = [
    "runtime",
    "budget",
    "genres_list",
    "director",
    "top_cast",
    "keywords_list",
    "release_month",
]

# define numeric features
numeric_features = ["runtime", "budget"]

# Define categorical features
category_features = ["director", "release_month"]

# set roi model data = movies without target column.
roi_model_data = movies_abt.dropna(subset=[target]).copy()

print(f"modeling rows: {len(roi_model_data):,}")

print(roi_model_data[target].value_counts())

print("roi_model_data baseline")

# Compare % of of all data that passes vs doesn't pass roi threshold.
print(roi_model_data[target].value_counts(normalize=True).round(4))

# Majority share = targets that = 1.
majority_share = roi_model_data[target].value_counts(normalize=True).max()
print(f"baseline accuracy (always predict majority class): {majority_share:.1%}")

# y is the target in roi_modeldata
y = roi_model_data[target]

# Create dummy columns and seperate cast, keywords, and genres lists.
X_cast = roi_model_data["top_cast"].str.get_dummies(sep="|").add_prefix("cast_")
X_keywords = roi_model_data["keywords_list"].str.get_dummies(sep="|").add_prefix("kw_")
X_genres = roi_model_data["genres_list"].str.get_dummies(sep="|").add_prefix("genre_")

# single-value categories
X_cat = pd.get_dummies(roi_model_data[category_features].astype(str),
                       drop_first=True, dtype=int)

# combine category colmns with numberic columns.
X = pd.concat([roi_model_data[numeric_features], X_cat, X_genres, X_cast, X_keywords], axis=1)
print(X.shape)

print(X.columns[:5].tolist(), X.columns[-5:].tolist())

modeling rows: 3,734
roi_threshold_met
1    2158
0    1576
Name: count, dtype: int64
roi_model_data baseline
roi_threshold_met
1    0.5779
0    0.4221
Name: proportion, dtype: float64
baseline accuracy (always predict majority class): 57.8%
(3734, 12045)
['runtime', 'budget', 'director_Abel Ferrara', 'director_Adam Elliot', 'director_Adam Marcus'] ['kw_воин', 'kw_любовь', 'kw_卧底肥妈', 'kw_绝地奶霸', 'kw_超级妈妈']


### Splitting The Data

Since our target is categorical we will use a logistic regression to score each movie based on it's probability of financial success using the features. 

A 60/20/20 split allows us to train, validate, and test our model seperately.

Stratification ensures that all sets contain the same distribution of target classes.

In [110]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

# X and y line up before splitting
print(len(X), len(y), X.index.equals(y.index))

# split sizes add back to the total
print(f"train {len(X_train):,} / valid {len(X_valid):,} / test {len(X_test):,} "
      f"= {len(X_train) + len(X_valid) + len(X_test):,} (total {len(X):,})")

# stratify kept the class balance
print(y_train.mean().round(3), y_valid.mean().round(3), y_test.mean().round(3))

3734 3734 True
train 2,240 / valid 747 / test 747 = 3,734 (total 3,734)
0.578 0.578 0.578


### Creating the baseline

Before we continue modeling, let's establish a baseline estimate that we can use as a benchmark for future models.

The baseline is that all movies are financially successful.

In [ ]:
def classification_metrics(model_name, dataset, actual, predicted):
    """Return accuracy, precision, recall, flag rate, and error counts as a dictionary."""
    # Unpack the four cells of the 2x2 confusion matrix
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "dataset": dataset,
        "accuracy": accuracy_score(actual, predicted),
        "precision": precision_score(actual, predicted, zero_division=0),
        "recall": recall_score(actual, predicted),
        "flag_rate": np.mean(predicted),
        "false_success": fp,
        "missed_success": fn,
    }


baseline_valid_pred = np.ones(len(y_valid), dtype=int) 

pd.DataFrame([
    classification_metrics("Baseline: always success", "validation", y_valid, baseline_valid_pred),
]).round(3)

,model,dataset,accuracy,precision,recall,flag_rate,false_success,missed_success
0,Baseline: always success,validation,0.578,0.578,1.0,1.0,315,0


sadfas

In [120]:
log_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
log_model.fit(X_train, y_train)
log_valid_prob = log_model.predict_proba(X_valid)[:, 1]
log_valid_pred = (log_valid_prob >= 0.5).astype(int)

tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)
tree_valid_pred = tree_model.predict(X_valid)

pd.DataFrame([
    classification_metrics("Baseline: always success", "validation", y_valid, baseline_valid_pred),
    classification_metrics("Logistic regression", "validation", y_valid, log_valid_pred),
    classification_metrics("Decision tree (max_depth=3)", "validation", y_valid, tree_valid_pred),
]).round(3)

,model,dataset,accuracy,precision,recall,flag_rate,false_success,missed_success
0,Baseline: always success,validation,0.578,0.578,1.000,1.000,315,0
1,Logistic regression,validation,0.589,0.632,0.694,0.636,175,132
2,Decision tree (max_depth=3),validation,0.620,0.611,0.944,0.894,260,24


In [119]:
rows = []
for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    pred = (log_valid_prob >= threshold).astype(int)
    rows.append(classification_metrics(f"Logistic, threshold {threshold:.2f}", "validation", y_valid, pred))

threshold_table = pd.DataFrame(rows).round(3)
display(threshold_table)

,model,dataset,accuracy,precision,recall,flag_rate,false_success,missed_success
0,"Logistic, threshold 0.20",validation,0.600,0.618,0.808,0.756,216,83
1,"Logistic, threshold 0.25",validation,0.594,0.617,0.785,0.735,210,93
2,"Logistic, threshold 0.30",validation,0.586,0.616,0.757,0.711,204,105
3,"Logistic, threshold 0.35",validation,0.589,0.621,0.743,0.692,196,111
4,"Logistic, threshold 0.40",validation,0.582,0.619,0.722,0.675,192,120
5,"Logistic, threshold 0.45",validation,0.588,0.627,0.708,0.653,182,126
6,"Logistic, threshold 0.50",validation,0.589,0.632,0.694,0.636,175,132


### Try model with different features.

In [131]:
# Final Practice Step 1
# Create evening_columns: every column in inflight_columns except "dep_delay".
# Hint: [col for col in inflight_columns if col != "dep_delay"]

no_kw_column = [col for col in X if col != "dep_delay"]

display(no_kw_column)

# combine category colmns with numberic columns.
X_less_features = pd.concat([roi_model_data[numeric_features], X_cat, X_genres], axis=1)
print(X_less_features.shape)

['runtime',
 'budget',
 'director_Abel Ferrara',
 'director_Adam Elliot',
 'director_Adam Marcus',
 'director_Adam McKay',
 'director_Adam Rifkin',
 'director_Adam Shankman',
 'director_Adam Simon',
 'director_Adam Wingard',
 'director_Adrian Lyne',
 'director_Adrienne Shelly',
 'director_Aki Kaurismäki',
 'director_Akira Kurosawa',
 'director_Akiva Goldsman',
 'director_Akiva Schaffer',
 'director_Alan J. Pakula',
 'director_Alan Parker',
 'director_Alan Poul',
 'director_Alan Shapiro',
 'director_Alan Taylor',
 'director_Albert Hughes',
 'director_Albert Magnoli',
 'director_Alejandro Agresti',
 'director_Alejandro Amenábar',
 'director_Alejandro González Iñárritu',
 'director_Aleksander Bach',
 'director_Alex Cox',
 'director_Alex Garland',
 'director_Alex Gibney',
 'director_Alex Kendrick',
 'director_Alex Kurtzman',
 'director_Alex Proyas',
 'director_Alex Rivera',
 'director_Alexander Payne',
 'director_Alexander Witt',
 'director_Alexandre Aja',
 'director_Alfonso Aráu',
 'direc

(3734, 1569)


In [ ]:
# Final Practice Step 1
# Create evening_columns: every column in inflight_columns except "dep_delay".
# Hint: [col for col in inflight_columns if col != "dep_delay"]

# set feature columns
feature_columns = [
    "runtime",
    "budget",
    "genres_list",
    "director",
    "top_cast",
    "keywords_list",
    "release_month",
]

# define numeric features
numeric_features = ["runtime", "budget"]

# Define categorical features
category_features = ["director", "release_month"]


X_small = pd.concat([
    roi_model_data[numeric_features],
    pd.get_dummies(roi_model_data[["primary_genre", "release_month"]].astype(str), drop_first=True, dtype=int),
], axis=1)

X_small_train = X_small.loc[X_train.index]
X_small_valid = X_small.loc[X_valid.index]
print(X.shape, X_small.shape)

(3734, 12045) (3734, 30)


In [124]:
display(movies_abt.columns)

print(X.isna().sum().sum()) 

Index(['movieId', 'tmdbId', 'imdb_id', 'title', 'original_language',
       'release_date', 'runtime', 'budget', 'revenue', 'vote_average',
       'vote_count', 'genres_list', 'primary_genre', 'release_year', 'roi',
       'roi_threshold_met', 'rating_count', 'avg_rating', 'min_rating',
       'max_rating', 'rating_std_dev', 'distinct_users', 'min_rating_date',
       'max_rating_date', 'rating_threshold_met', 'director', 'top_cast',
       'keywords_list', 'release_month'],
      dtype='str')

0


In [125]:
movies_abt.head(3)

,movieId,tmdbId,imdb_id,title,original_language,release_date,runtime,budget,revenue,vote_average,vote_count,genres_list,primary_genre,release_year,roi,roi_threshold_met,rating_count,avg_rating,min_rating,max_rating,rating_std_dev,distinct_users,min_rating_date,max_rating_date,rating_threshold_met,director,top_cast,keywords_list,release_month
0,1,862,tt0114709,Toy Story,en,1995-10-30,81.0,30000000.0,373554033.0,7.7,5415,Animation|Comedy|Family,Animation,1995,11.451801,1,247.0,3.872470,1.0,5.0,0.958981,247.0,1996-03-30 19:00:13,2016-10-06 19:55:11,0.0,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,10
1,2,8844,tt0113497,Jumanji,en,1995-12-15,104.0,65000000.0,262797249.0,6.9,2413,Adventure|Fantasy|Family,Adventure,1995,3.043035,1,107.0,3.401869,1.5,5.0,0.880714,107.0,1996-03-30 19:12:30,2016-08-01 17:42:33,0.0,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,12
2,4,31357,tt0114885,Waiting to Exhale,en,1995-12-22,127.0,16000000.0,81452156.0,6.1,34,Comedy|Drama|Romance,Comedy,1995,4.090760,1,13.0,2.384615,1.0,3.5,0.938835,13.0,1996-06-10 16:45:35,2004-07-27 06:14:12,0.0,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,12
